# AlegroCode — Тест пайплайна (без сервера)

Этот ноутбук прогоняет весь ML-пайплайн напрямую, **без запуска сервера и клиента**.
Удобен для отладки моделей, проверки форматов данных и визуализации результатов.

**Порядок запуска:** 1 → 2 → 3 → 4 → 5

| Ячейка | Что делает |
|--------|------------|
| 1 — Setup | Устанавливает зависимости и настраивает окружение |
| 2 — Load models | Загружает все ML-модели в память |
| 3 — Load image | Загружает тестовое изображение (укажите путь или URL) |
| 4 — Run pipeline | Запускает полный анализ и показывает результаты |
| 5 — Inspect results | Детальный просмотр всех полей ответа |

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1 — SETUP                                                      ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, sys, textwrap
from pathlib import Path

repo_root = Path('/work/building_analyzer')
if not repo_root.exists():
    repo_root = Path.cwd().resolve()
os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Корень репозитория:', repo_root)

os.environ.pop('PIP_CONSTRAINT', None)

(repo_root / 'constraints-colab.txt').write_text(textwrap.dedent("""\
    numpy>=1.26,<2.0
    opencv-python-headless==4.10.0.84
    matplotlib>=3.8,<3.11
    packaging<=24.2
    transformers>=4.44,<4.46
    accelerate>=0.33,<0.35
    tokenizers>=0.19,<0.20
    huggingface-hub>=0.24,<1.0
    tqdm>=4.66,<5
"""), encoding='utf-8')

src_req = repo_root / 'backend' / 'requirements.txt'
flt_req = repo_root / 'backend' / 'requirements.colab.filtered.txt'
skip = ('torch','torchvision','transformers','accelerate','tokenizers',
        'huggingface-hub','numpy','opencv-python-headless','matplotlib')
lines = []
for line in src_req.read_text(encoding='utf-8').splitlines():
    s = line.strip()
    if not s or s.startswith('#') or not s.startswith(skip):
        lines.append(line)
flt_req.write_text('\n'.join(lines) + '\n', encoding='utf-8')

%pip install -q -U "pip<26.1" setuptools wheel
%pip install -q --prefer-binary -c constraints-colab.txt \
    "numpy>=1.26,<2.0" "opencv-python-headless==4.10.0.84" "matplotlib>=3.8,<3.11" \
    "transformers>=4.44,<4.46" "accelerate>=0.33,<0.35" \
    "tokenizers>=0.19,<0.20" "huggingface-hub>=0.24,<1.0"
%pip install -q --prefer-binary \
    -r backend/requirements.colab.filtered.txt -c constraints-colab.txt
%pip install -q --prefer-binary -c constraints-colab.txt \
    python-multipart "sqlalchemy>=2,<3" aiosqlite "pydantic-settings>=2,<3"
%pip install -q --no-build-isolation --no-deps \
    git+https://github.com/facebookresearch/sam2.git

import numpy, cv2
print(f"numpy {numpy.__version__} | cv2 {cv2.__version__}")
print("✅  Setup complete")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2 — LOAD MODELS                                                ║
# ╚══════════════════════════════════════════════════════════════════╝
# Модели загружаются в глобальную переменную `analyzer`.
# Ячейку достаточно запустить один раз — при повторном запуске
# она обнаружит `analyzer` в globals() и пропустит загрузку.

import time

if 'analyzer' not in globals():
    from backend.ml_pipeline import FacadeAnalyzer
    t0 = time.time()
    analyzer = FacadeAnalyzer()
    analyzer.load_models()
    print(f'Модели загружены за {time.time()-t0:.1f}s, устройство: {analyzer.device}')
else:
    print(f'Модели уже загружены (device={analyzer.device})')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3 — LOAD IMAGE                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
# Укажите путь к изображению фасада или URL.
# Если файла нет — создаётся синтетическое тестовое изображение.

import urllib.request
import numpy as np
import cv2
import matplotlib.pyplot as plt

# ─── НАСТРОЙТЕ ЭТО ────────────────────────────────────────────────────────
IMAGE_SOURCE = None   # путь к файлу (str) или URL (str) или None → синтетика
# Пример: IMAGE_SOURCE = '/content/my_building.jpg'
# Пример: IMAGE_SOURCE = 'https://example.com/facade.jpg'
# ──────────────────────────────────────────────────────────────────────────

if IMAGE_SOURCE and IMAGE_SOURCE.startswith('http'):
    print('Скачиваем изображение...')
    with urllib.request.urlopen(IMAGE_SOURCE) as resp:
        image_bytes = resp.read()
    print(f'Загружено {len(image_bytes):,} байт')
elif IMAGE_SOURCE:
    image_bytes = open(IMAGE_SOURCE, 'rb').read()
    print(f'Прочитано {len(image_bytes):,} байт из {IMAGE_SOURCE}')
else:
    print('IMAGE_SOURCE не задан — создаём синтетическое изображение фасада')
    # Синтетический фасад: кирпичная стена 600×800 с окнами и трещиной
    h, w = 600, 800
    img = np.full((h, w, 3), [180, 160, 140], dtype=np.uint8)  # бежевая стена
    # Кирпичная текстура
    for row in range(0, h, 30):
        for col in range(0, w, 60):
            offset = 30 if (row // 30) % 2 else 0
            x = (col + offset) % w
            cv2.rectangle(img, (x, row), (x+58, row+28),
                          (160, 130, 110), -1)
            cv2.rectangle(img, (x, row), (x+58, row+28),
                          (100, 80, 70), 1)
    # Окна
    for wx, wy in [(100, 100), (300, 100), (500, 100), (100, 350), (500, 350)]:
        cv2.rectangle(img, (wx, wy), (wx+150, wy+200), (180, 220, 240), -1)
        cv2.rectangle(img, (wx, wy), (wx+150, wy+200), (80, 80, 80), 3)
        cv2.line(img, (wx+75, wy), (wx+75, wy+200), (80,80,80), 2)
        cv2.line(img, (wx, wy+100), (wx+150, wy+100), (80,80,80), 2)
    # Трещина (дефект)
    pts = np.array([[350,200],[355,250],[345,300],[360,380],[350,450]], np.int32)
    cv2.polylines(img, [pts], False, (60,50,45), 3)
    # Пятно влаги
    cv2.ellipse(img, (620, 300), (80, 60), 0, 0, 360, (140, 150, 160), -1)
    _, image_bytes = cv2.imencode('.jpg', img, [cv2.IMWRITE_JPEG_QUALITY, 90])
    image_bytes = bytes(image_bytes)
    print(f'Синтетическое изображение: {w}x{h}px, {len(image_bytes):,} байт')

# Показываем исходник
nparr = np.frombuffer(image_bytes, np.uint8)
img_show = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
img_show_rgb = cv2.cvtColor(img_show, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 7))
plt.imshow(img_show_rgb)
plt.title(f'Тестовое изображение  {img_show.shape[1]}×{img_show.shape[0]}px')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4 — RUN PIPELINE                                               ║
# ╚══════════════════════════════════════════════════════════════════╝
import time
import traceback
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

OUTPUT_DIR = '/tmp/pipeline_test'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print('▶ Запускаем analyzer.analyze()...')
t0 = time.time()
try:
    result = analyzer.analyze(image_bytes, output_dir=OUTPUT_DIR)
    elapsed = time.time() - t0
    print(f'✅  Анализ завершён за {elapsed:.1f}s')
except Exception:
    print('❌  Ошибка в пайплайне:')
    traceback.print_exc()
    raise

# ── Сводка ────────────────────────────────────────────────────────────────
print()
print('═' * 55)
print(f'  ID             : {result["id"]}')
print(f'  Оценка         : {result["overall_score"]:.1f} / 100')
print(f'  Состояние      : {result["overall_condition"]}')
print(f'  Площадь (px)   : {result["total_area_px"]:,}')
print(f'  Повреждено (px): {result["damaged_area_px"]:,}')
print('═' * 55)

print()
if result.get('damages'):
    print('Дефекты:')
    for d in result['damages']:
        print(f'  {d["type_display"]:30s} {d["percentage"]:5.1f}%  [{d["severity_display"]}]')
else:
    print('Дефектов не обнаружено')

print()
if result.get('materials'):
    print('Материалы:')
    for m in result['materials']:
        print(f'  {m["name_display"]:30s} {m["percentage"]:5.1f}%')

# ── Визуализации ──────────────────────────────────────────────────────────
viz_paths = result.get('image_paths', {})
if viz_paths:
    n = len(viz_paths)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    for ax, (name, path) in zip(axes, viz_paths.items()):
        img_v = cv2.imread(path)
        if img_v is not None:
            ax.imshow(cv2.cvtColor(img_v, cv2.COLOR_BGR2RGB))
        ax.set_title(name)
        ax.axis('off')
    for ax in axes[len(viz_paths):]:
        ax.set_visible(False)
    plt.suptitle('Визуализации пайплайна', fontsize=14)
    plt.tight_layout()
    plt.show()

# ── Маски дефектов ────────────────────────────────────────────────────────
defect_masks = (result.get('masks') or {}).get('defects') or {}
if defect_masks:
    n = len(defect_masks)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    for ax, (name, mask) in zip(axes, defect_masks.items()):
        if mask is not None:
            ax.imshow(mask.astype(np.uint8) * 255, cmap='hot')
        ax.set_title(f'defect: {name}')
        ax.axis('off')
    for ax in axes[len(defect_masks):]:
        ax.set_visible(False)
    plt.suptitle('Маски дефектов', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5 — INSPECT RESULTS                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
# Детальный просмотр всех полей результата без визуализации.
# Удобно для отладки и проверки структуры данных.

import json
import numpy as np

def _serialize(obj):
    """Сериализатор для numpy-объектов в JSON."""
    if isinstance(obj, np.ndarray):
        return f'<ndarray shape={obj.shape} dtype={obj.dtype}>'
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    raise TypeError(f'Not serializable: {type(obj)}')

# Показываем все поля кроме объёмных numpy-массивов
result_safe = {}
for k, v in result.items():
    if k == 'masks':
        # Показываем только наличие масок и их размеры
        masks_info = {}
        for cat, cat_masks in (v or {}).items():
            masks_info[cat] = {
                name: f'shape={m.shape} nonzero={int(np.count_nonzero(m))}'
                for name, m in (cat_masks or {}).items()
                if m is not None
            }
        result_safe[k] = masks_info
    else:
        result_safe[k] = v

print(json.dumps(result_safe, indent=2, ensure_ascii=False, default=_serialize))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6 — ESTIMATE (опционально)                                     ║
# ╚══════════════════════════════════════════════════════════════════╝
# Проверяет расчёт стоимости ремонта независимо от сервера.

import asyncio
import nest_asyncio
nest_asyncio.apply()

from backend.calibration import fallback_from_total_area
from backend.estimator.calculator import build_estimate_from_analysis

# Используем калибровку «по площади» (450 м² — типовой фасад)
cal = fallback_from_total_area(
    total_area_m2=450.0,
    total_area_px=result['total_area_px'] or 1
)

damages_m2 = []
for d in result['damages']:
    damages_m2.append({**d, 'area_m2': round(cal.area_px_to_m2(d['area_px']), 2)})

layer_m2 = {}
for k, v in (result.get('layer_analysis') or {}).items():
    layer_m2[k] = {**v, 'area_m2': round(cal.area_px_to_m2(v['area_px']), 2)}

estimate = asyncio.get_event_loop().run_until_complete(
    build_estimate_from_analysis(
        damages=damages_m2,
        layer_analysis=layer_m2,
        scale=cal,
    )
)

import json
print(json.dumps(estimate, indent=2, ensure_ascii=False))